# 04 — Prompt Services

Disentangling what a translation tells us about the **service** (model capability) vs. the **prompt** (framing effect) using a 2×2 analytical framework:

| **Within prompt** | **Across prompts** |
|---|---|
| **Within service** | §1 Fingerprint each (service, variant) as a unit | §3 Prompt stability — does one service agree with itself? |
| **Across services** | §2 Service agreement — do models converge on one prompt? | §4 Ultimate consensus — cross-service *and* cross-prompt signal |

**Exclusion tier: Tier 1 — Service Exploration** (see [docs/exclusion_strategy.md](../docs/exclusion_strategy.md))

At this tier, errors are treated as data: a service that hallucinates or produces mixed-script output is telling us something real about its coverage. The only automatic suppression is `enforce_translation_rationale_pairing` (applied at load time by `load_variant_df`). Manual `exclude_translation=True` entries are also applied — these represent structural incompatibilities (complete output failure, not quality concerns). All automated quality flags from `quality_flags.csv` are merged as annotation columns so they are visible in the analysis without filtering the data.

**Scripts:** `explore_confidence_within_variant.py` (→ `confidence_scores.csv`) and
`explore_confidence_across_variants.py` (→ `across_variant_detail.csv`).


In [1]:
import ast
import os
import sys
from collections import Counter

import pandas as pd
import altair as alt
from pathlib import Path

alt.data_transformers.enable('vegafusion')

sys.path.insert(0, str(Path('..').resolve()))
from scripts.utils import get_data_directory_path, read_csv_file, get_language_family
from scripts.exploration.explore_confidence_within_variant import (
    run_confidence_evaluation, ALL_VARIANTS,
)
from scripts.exploration.explore_confidence_across_variants import (
    run_across_variant_evaluation, LLM_SERVICES, BASELINE_SERVICES,
)

DATA_DIR  = get_data_directory_path()
TERM      = 'Digital Humanities'
TERM_SLUG = TERM.lower().replace(' ', '_')
EVAL_DIR  = os.path.join(DATA_DIR, 'translated_terms', TERM_SLUG, 'evaluation')
VARIANTS  = ALL_VARIANTS

VARIANT_LABELS = {
    'minimal':          'Minimal',
    'expert_persona':   'Expert Persona',
    'native_rationale': 'Native Rationale',
    'judge':            'Judge',
}
SERVICE_ORDER = ['Wikipedia', 'Google Translate', 'EasyNMT', 'Lingvanex',
                 'OpenAI', 'Claude', 'Gemini', 'DeepSeek',
                 'Llama', 'Gemma', 'Qwen', 'Mistral']
SERVICE_COLOURS = {
    'Wikipedia': '#aec7e8', 'Google Translate': '#c5b0d5',
    'EasyNMT': '#c49c94',   'Lingvanex': '#dbdb8d',
    'OpenAI': '#1f77b4',    'Claude': '#ff7f0e',
    'Gemini': '#2ca02c',    'DeepSeek': '#9467bd',
    'Llama': '#8c564b',     'Gemma': '#e377c2',
    'Qwen': '#7f7f7f',      'Mistral': '#bcbd22',
}
LLM_TRANS_COLS = {
    'Claude':   'claude_translated_term',
    'OpenAI':   'openai_translated_term',
    'Gemini':   'gemini_translated_term',
    'DeepSeek': 'deepseek_translated_term',
    'Llama':    'llama_translated_term',
    'Gemma':    'gemma_translated_term',
    'Qwen':     'qwen_translated_term',
    'Mistral':  'mistral_translated_term',
}

TIER_ORDER   = ['all_identical', 'trivial_only', 'has_content']
TIER_LABELS  = {
    'all_identical': 'All identical',
    'trivial_only':  'Trivial only (cap / whitespace)',
    'has_content':   'Content differences',
}
TIER_COLOURS = {'all_identical': '#2ca02c', 'trivial_only': '#ff7f0e', 'has_content': '#d62728'}

def classify_diff_tier(d):
    if pd.isna(d) or str(d) in ('no_differences', 'all_identical', 'unknown', ''):
        return 'all_identical'
    types = set(str(d).split(','))
    return 'trivial_only' if types <= {'capitalization', 'whitespace', 'both'} else 'has_content'

print(f'DATA_DIR: {DATA_DIR}')  
print(f'Variants: {VARIANTS}')

Retrieving translation pipeline data directory path...

DATA_DIR: /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets
Variants: ['minimal', 'expert_persona', 'native_rationale', 'judge']


In [2]:
# ── Within-variant confidence scores ─────────────────────────────────────────
scored_df, summary_df = run_confidence_evaluation(
    data_directory_path=DATA_DIR,
    target_terms=[TERM],
    variants=VARIANTS,
    output_dir=EVAL_DIR,
)
scored_df['language_family'] = scored_df['language_code'].apply(get_language_family)
print(f'scored_df: {len(scored_df)} rows')

# ── Across-variant detail ─────────────────────────────────────────────────────
detail_path  = os.path.join(EVAL_DIR, 'across_variant_detail.csv')
summary_path = os.path.join(EVAL_DIR, 'across_variant_service_summary.csv')
if not os.path.exists(detail_path):
    print('Running explore_confidence_across_variants.py...')
    detail_df, across_summary_df = run_across_variant_evaluation(DATA_DIR, [TERM])
else:
    detail_df       = read_csv_file(detail_path)
    across_summary_df = read_csv_file(summary_path)

has_data = detail_df[detail_df['n_variants_present'] > 0].copy()
llm_df   = has_data[~has_data['is_baseline']].copy()
print(f'detail_df: {len(detail_df)} rows | llm_df: {len(llm_df)} rows')
summary_df

📊 Scoring: Digital Humanities

  ✓ Loaded 880 rows for variant 'minimal'

  ✓ Loaded 880 rows for variant 'expert_persona'

  ✓ Loaded 880 rows for variant 'native_rationale'

  ✓ Loaded 880 rows for variant 'judge'

...

  confidence_summary.csv : 4 rows

                              Confidence Scoring Summary (LLM agreement per variant)                               
┏━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ term       ┃ variant    ┃ total_lan… ┃ language… ┃ mean_llm_… ┃ median_l… ┃ std_llm_c… ┃ cv_llm_c… ┃ llm_uniqu… ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ Digital    │ minimal    │ 880        │ 880       │ 0.2299     │ 0.1429    │ 0.1537     │ 0.6685    │ 6.85       │
│ Humanities │            │            │           │            │           │            │           │            │
│ Digital    │ expert_pe… │ 880        │ 880       │ 0.2035     │ 0.125     │ 0.1452     │ 0.7138    │ 7.21       │
│ Humanities │            │            │           │            │           │            │           │            │
│ Digital    │ native_ra… │ 880        │ 880       │ 0.211      │ 0.1429    │ 0.1445     │ 0.6846    │ 6.88       │
│ Humanities │            │            │           │            │           │            │           │            │
│ Digital    │ judge      │ 880        │ 880       │ 0.4671     │ 0.375     │ 0.212      │ 0.4539    │ 4.59       │
│ Humanities │            │            │           │            │           │            │           │            │
└────────────┴────────────┴────────────┴───────────┴────────────┴───────────┴────────────┴───────────┴────────────┘

Retrieving translation pipeline data directory path...

scored_df: 3520 rows
detail_df: 10560 rows | llm_df: 7040 rows


,term,variant,total_languages,languages_with_llm,mean_llm_confidence,median_llm_confidence,std_llm_confidence,cv_llm_confidence,llm_unique_candidates_mean,languages_with_baseline,mean_baseline_confidence
0,Digital Humanities,minimal,880,880,0.2299,0.1429,0.1537,0.6685,6.85,257,0.7662
1,Digital Humanities,expert_persona,880,880,0.2035,0.1250,0.1452,0.7138,7.21,257,0.7662
2,Digital Humanities,native_rationale,880,880,0.2110,0.1429,0.1445,0.6846,6.88,257,0.7662
3,Digital Humanities,judge,880,880,0.4671,0.3750,0.2120,0.4539,4.59,257,0.7662


In [3]:
# ── Tier 1 exclusion policy ──────────────────────────────────────────────────
# Merge quality flags as annotations (never filter on them at Tier 1).
# Apply only exclude_translation=True from manual_exclusions — structural drops only.

from scripts.exploration.translation_classifier import curate_translation

flags_path = os.path.join(EVAL_DIR, 'quality_flags.csv')
quality_flags = read_csv_file(flags_path)

FLAG_ANNOTATION_COLS = [
    'language_code',
    'has_mixed_script', 'has_placeholder_term', 'has_romanization',
    'has_source_term', 'has_script_disagreement', 'has_repetition_loop',
    'has_extreme_term_length', 'has_unicode_escape', 'has_any_mixing',
    'quality_flags', 'flag_count',
]
scored_df = scored_df.merge(quality_flags[FLAG_ANNOTATION_COLS], on='language_code', how='left')

excl_path = os.path.join(EVAL_DIR, 'manual_exclusions.csv')
if os.path.exists(excl_path):
    excl_df = read_csv_file(excl_path)
    whole_drops = excl_df[excl_df['exclude_translation'] == True][['language_code', 'service']]
    tier1_drop_pairs = set(zip(whole_drops['language_code'], whole_drops['service']))

    # Languages where ALL eight LLM services are dropped → no usable LLM data
    LLM_NAMES = set(LLM_TRANS_COLS.keys())
    drops_by_lang = whole_drops.groupby('language_code')['service'].apply(set).to_dict()
    fully_excluded = {lc for lc, svcs in drops_by_lang.items() if LLM_NAMES <= svcs}
    partially_excluded = {lc for lc in drops_by_lang if lc not in fully_excluded}

    # Remove fully-excluded languages; flag partial exclusions as annotation
    scored_df = scored_df[~scored_df['language_code'].isin(fully_excluded)].copy()
    scored_df['has_tier1_drop'] = scored_df['language_code'].isin(partially_excluded)

    # Re-derive detail_df and llm_df after exclusions
    detail_df = detail_df[~detail_df['language_code'].isin(fully_excluded)].copy()
    has_data = detail_df[detail_df['n_variants_present'] > 0].copy()
    llm_df   = has_data[~has_data['is_baseline']].copy()

    print(f'Tier 1 exclusions:')
    print(f'  Whole-translation drops  : {len(whole_drops)} (language × service) pairs')
    print(f'  Fully excluded languages : {len(fully_excluded)} (all LLM services dropped)')
    print(f'  Partially excluded       : {len(partially_excluded)} (flagged; some services remain)')
    print(f'  Annotated quality flags  : {int(scored_df["flag_count"].gt(0).sum())} languages with ≥1 flag (retained as data)')
else:
    scored_df['has_tier1_drop'] = False
    print('No manual_exclusions.csv found — skipping Tier 1 drops')


Tier 1 exclusions:
  Whole-translation drops  : 107 (language × service) pairs
  Fully excluded languages : 0 (all LLM services dropped)
  Partially excluded       : 86 (flagged; some services remain)
  Annotated quality flags  : 2792 languages with ≥1 flag (retained as data)


## §1 — Service × Prompt Fingerprinting
_Within service · within prompt_

With one run per (service, variant) combination there is no within-cell agreement to measure, so this section characterises each of the 16 (service × prompt) combinations as a unit: how many languages does it cover, and what does its output profile look like? This is the baseline before any cross-service or cross-prompt comparison.

In [4]:
llm_only = {s: c for s, c in LLM_TRANS_COLS.items()}
fingerprint_rows = []
for variant in VARIANTS:
    vdf = scored_df[scored_df['prompt_variant'] == variant]
    for svc, col in llm_only.items():
        if col not in vdf.columns:
            continue
        vals = vdf[col].dropna()
        vals = vals[~vals.astype(str).str.strip().isin(['', 'nan'])]
        n_langs    = len(vals)
        mean_wc    = vals.astype(str).str.split().str.len().mean() if n_langs else 0
        # Confidence when this service is the only one being compared to itself
        svc_conf   = vdf.loc[vals.index, 'llm_confidence'].mean() if n_langs else float('nan')
        fingerprint_rows.append({
            'service': svc, 'variant': variant,
            'n_languages': n_langs,
            'mean_word_count': round(mean_wc, 2) if n_langs else 0,
            'mean_llm_confidence': round(svc_conf, 3) if n_langs else float('nan'),
        })

fp_df = pd.DataFrame(fingerprint_rows)

llm_order = ['Claude', 'OpenAI', 'Gemini', 'DeepSeek', 'Llama', 'Gemma', 'Qwen', 'Mistral']

cov_heat = alt.Chart(fp_df).mark_rect().encode(
    x=alt.X('variant:N', sort=VARIANTS, title=None,
             axis=alt.Axis(labelAngle=-20)),
    y=alt.Y('service:N', sort=llm_order, title=None),
    color=alt.Color('n_languages:Q', title='languages covered',
                    scale=alt.Scale(scheme='blues')),
    tooltip=['service:N', 'variant:N', 'n_languages:Q',
             alt.Tooltip('mean_word_count:Q', format='.1f'),
             alt.Tooltip('mean_llm_confidence:Q', format='.3f')],
).properties(width=260, height=160, title='Coverage (n languages)')
cov_text = cov_heat.mark_text(fontSize=10).encode(
    text='n_languages:Q',
    color=alt.condition(alt.datum.n_languages > 800,
                        alt.value('white'), alt.value('black')))

wc_heat = alt.Chart(fp_df).mark_rect().encode(
    x=alt.X('variant:N', sort=VARIANTS, title=None,
             axis=alt.Axis(labelAngle=-20)),
    y=alt.Y('service:N', sort=llm_order, title=None),
    color=alt.Color('mean_word_count:Q', title='mean word count',
                    scale=alt.Scale(scheme='oranges')),
    tooltip=['service:N', 'variant:N',
             alt.Tooltip('mean_word_count:Q', format='.2f')],
).properties(width=260, height=160, title='Mean word count')
wc_text = wc_heat.mark_text(fontSize=10).encode(
    text=alt.Text('mean_word_count:Q', format='.1f'),
    color=alt.condition(alt.datum.mean_word_count > 3.5,
                        alt.value('white'), alt.value('black')))

(cov_heat + cov_text) | (wc_heat + wc_text)

alt.HConcatChart(...)

## §2 — Within Prompt × Across Services
_Cross-service agreement per prompt variant_

For each prompt variant independently: how much do the eight LLM services agree on the same translation? High agreement means all models converge on the same answer when asked the same way. Low agreement means models have genuinely different outputs regardless of the prompt.

Key metric: **LLM confidence** = fraction of services agreeing on the most common translation within a (language, variant).

In [5]:
plot_data = scored_df[scored_df['llm_total_services'] > 0].copy()

box = alt.Chart(plot_data).mark_boxplot(extent='min-max').encode(
    x=alt.X('prompt_variant:N', sort=VARIANTS, title=None),
    y=alt.Y('llm_confidence:Q', scale=alt.Scale(domain=[0, 1]),
            title='LLM confidence'),
    color=alt.Color('prompt_variant:N', sort=VARIANTS, legend=None),
    tooltip=['prompt_variant:N', 'llm_confidence:Q'],
).properties(width=300, height=280,
             title='LLM confidence distribution per variant')

cands = (plot_data.groupby('prompt_variant')['llm_unique_candidates']
         .mean().reset_index()
         .rename(columns={'llm_unique_candidates': 'mean_unique_candidates'}))
bar = alt.Chart(cands).mark_bar().encode(
    x=alt.X('prompt_variant:N', sort=VARIANTS, title=None),
    y=alt.Y('mean_unique_candidates:Q', title='mean unique candidates',
            scale=alt.Scale(domain=[0, 4])),
    color=alt.Color('prompt_variant:N', sort=VARIANTS, legend=None),
    tooltip=['prompt_variant:N',
             alt.Tooltip('mean_unique_candidates:Q', format='.2f')],
).properties(width=300, height=280,
             title='Mean unique candidates (4 = all models differ)')
bar_text = bar.mark_text(dy=-8, fontSize=10).encode(
    text=alt.Text('mean_unique_candidates:Q', format='.2f'))

box | (bar + bar_text)

alt.HConcatChart(...)

In [6]:
heatmap_data = (
    scored_df[scored_df['llm_total_services'] > 0]
    .groupby(['language_family', 'prompt_variant'])['llm_confidence']
    .mean().reset_index().rename(columns={'llm_confidence': 'mean_confidence'})
)
family_order = (
    heatmap_data.groupby('language_family')['mean_confidence']
    .mean().sort_values(ascending=False).index.tolist()
)

rect = alt.Chart(heatmap_data).mark_rect().encode(
    x=alt.X('prompt_variant:N', sort=VARIANTS, title=None),
    y=alt.Y('language_family:N', sort=family_order, title=None),
    color=alt.Color('mean_confidence:Q',
                    scale=alt.Scale(scheme='redyellowgreen', domain=[0, 1]),
                    title='mean LLM confidence'),
    tooltip=['language_family:N', 'prompt_variant:N',
             alt.Tooltip('mean_confidence:Q', format='.2f')],
)
rect_text = rect.mark_text(fontSize=9).encode(
    text=alt.Text('mean_confidence:Q', format='.2f'),
    color=alt.condition(alt.datum.mean_confidence > 0.5,
                        alt.value('black'), alt.value('white')))

(rect + rect_text).properties(
    width=350, height=500,
    title='Mean LLM confidence — language family × prompt variant')

alt.LayerChart(...)

In [7]:
fully_divergent = (
    scored_df[
        (scored_df['llm_total_services'] >= 4) &
        (scored_df['llm_confidence'] <= 0.25)
    ]
    .groupby('language_code').size()
    .reset_index(name='variants_fully_divergent')
    .sort_values('variants_fully_divergent', ascending=False)
)
print(f'Fully divergent in ≥1 variant:  {len(fully_divergent)}')
print(f'Fully divergent in all 4 variants: '
      f'{(fully_divergent["variants_fully_divergent"] == 4).sum()}')

alt.Chart(fully_divergent.head(30)).mark_bar().encode(
    y=alt.Y('language_code:N', sort='-x', title=None),
    x=alt.X('variants_fully_divergent:Q',
            title='variants where LLMs are highly divergent (<=25% agree)',
            scale=alt.Scale(domain=[0, 4])),
    tooltip=['language_code:N', 'variants_fully_divergent:Q'],
).properties(width=350, height=500,
             title='Most consistently divergent languages (top 30)')

Fully divergent in ≥1 variant:  766
Fully divergent in all 4 variants: 224


alt.Chart(...)

In [8]:
comp_mean = (
    scored_df[scored_df['llm_total_services'] > 0]
    .groupby('prompt_variant')[['llm_confidence', 'baseline_confidence']]
    .mean().reindex(VARIANTS).reset_index()
    .melt(id_vars='prompt_variant', var_name='source', value_name='mean_confidence')
)
comp_mean['source'] = comp_mean['source'].map({
    'llm_confidence':      'LLM (varies per variant)',
    'baseline_confidence': 'Baseline (prompt-invariant)',
})

alt.Chart(comp_mean).mark_bar().encode(
    x=alt.X('prompt_variant:N', sort=VARIANTS, title=None),
    y=alt.Y('mean_confidence:Q', title='Mean confidence',
            scale=alt.Scale(domain=[0, 1])),
    xOffset='source:N',
    color=alt.Color('source:N', title=None),
    tooltip=['prompt_variant:N', 'source:N',
             alt.Tooltip('mean_confidence:Q', format='.3f')],
).properties(width=400, height=280,
             title='LLM vs baseline confidence per variant (baseline = sanity check)')

alt.Chart(...)

In [9]:
diff_df = scored_df[scored_df['llm_total_services'] >= 2].copy()
diff_df['diff_tier']  = diff_df['llm_difference_types'].apply(classify_diff_tier)
diff_df['tier_label'] = diff_df['diff_tier'].map(TIER_LABELS)

tier_counts = (diff_df['diff_tier'].value_counts()
               .reset_index().rename(columns={'count': 'n'}))
tier_counts.columns = ['diff_tier', 'n']
tier_counts['pct'] = (tier_counts['n'] / len(diff_df) * 100).round(1)
tier_counts['label'] = tier_counts['diff_tier'].map(TIER_LABELS)
for _, r in tier_counts.sort_values('diff_tier').iterrows():
    print(f"  {r['label']}: {r['n']} ({r['pct']}%)")

tier_bar = alt.Chart(tier_counts).mark_bar().encode(
    x='n:Q',
    y=alt.Y('label:N', sort=[TIER_LABELS[t] for t in TIER_ORDER], title=None),
    color=alt.Color('diff_tier:N',
        scale=alt.Scale(domain=TIER_ORDER,
                        range=[TIER_COLOURS[t] for t in TIER_ORDER]),
        legend=None),
).properties(width=320, height=120, title='Difference tier breakdown — within prompt')

non_identical = diff_df[diff_df['diff_tier'] != 'all_identical'].copy()
conf_box = alt.Chart(non_identical).mark_boxplot(extent='min-max').encode(
    x=alt.X('llm_confidence:Q', scale=alt.Scale(domain=[0, 1]),
            title='LLM confidence'),
    y=alt.Y('tier_label:N',
            sort=[TIER_LABELS['trivial_only'], TIER_LABELS['has_content']],
            title=None),
    color=alt.Color('diff_tier:N',
        scale=alt.Scale(domain=TIER_ORDER,
                        range=[TIER_COLOURS[t] for t in TIER_ORDER]),
        legend=None),
).properties(width=320, height=120,
             title='Confidence by tier (excl. all-identical)')

tier_bar & conf_box

  All identical: 32 (0.9%)
  Content differences: 3478 (98.8%)
  Trivial only (cap / whitespace): 10 (0.3%)


alt.VConcatChart(...)

## §3 — Across Prompts × Within Service
_Prompt stability per service_

For each service independently: does it produce the same translation regardless of how it was prompted? A service with high stability has a grounded answer. A service with low stability is prompt-sensitive — its output is more a reflection of the framing than the underlying language.

Key metric: **agreement_rate** = fraction of prompt variants that produced the same best translation for a given (language × service).

In [10]:
summary_plot = across_summary_df.copy()
summary_plot['is_baseline'] = summary_plot['is_baseline'].astype(bool)

bars = alt.Chart(summary_plot).mark_bar().encode(
    x=alt.X('mean_agreement:Q', scale=alt.Scale(domain=[0, 1]),
            title='Mean agreement rate across variants'),
    y=alt.Y('service:N', sort=SERVICE_ORDER, title=None),
    color=alt.Color('service:N',
        scale=alt.Scale(domain=list(SERVICE_COLOURS.keys()),
                        range=list(SERVICE_COLOURS.values())),
        legend=None),
    opacity=alt.condition(alt.datum.is_baseline,
                          alt.value(0.4), alt.value(1.0)),
    tooltip=['service:N', 'mean_agreement:Q', 'std_agreement:Q',
             'n_languages:Q'],
)
text = alt.Chart(summary_plot).mark_text(align='left', dx=4, fontSize=11).encode(
    x='mean_agreement:Q', y=alt.Y('service:N', sort=SERVICE_ORDER),
    text=alt.Text('mean_agreement:Q', format='.2f'))

(bars + text).properties(width=450, height=280,
    title='Service prompt stability — mean cross-variant agreement (baselines faded)')

alt.LayerChart(...)

In [11]:
llm_service_order = [s for s in SERVICE_ORDER if s in LLM_SERVICES]

alt.Chart(llm_df).mark_boxplot(extent='min-max', size=20).encode(
    x=alt.X('agreement_rate:Q', scale=alt.Scale(domain=[0, 1]),
            title='Agreement rate (per language)'),
    y=alt.Y('service:N', sort=llm_service_order, title=None),
    color=alt.Color('service:N',
        scale=alt.Scale(domain=list(SERVICE_COLOURS.keys()),
                        range=list(SERVICE_COLOURS.values())),
        legend=None),
).properties(width=450, height=200,
    title='Per-language agreement rate distribution (LLM services only)')

alt.Chart(...)

In [12]:
family_service = (
    llm_df.groupby(['language_family', 'service'])['agreement_rate']
    .mean().reset_index().rename(columns={'agreement_rate': 'mean_agreement'})
)
fam_order_s3 = (
    family_service.groupby('language_family')['mean_agreement']
    .mean().sort_values(ascending=False).index.tolist()
)

alt.Chart(family_service).mark_rect().encode(
    x=alt.X('service:N', sort=llm_service_order, title=None),
    y=alt.Y('language_family:N', sort=fam_order_s3, title='Language Family'),
    color=alt.Color('mean_agreement:Q',
                    scale=alt.Scale(scheme='blues', domain=[0, 1]),
                    title='Mean agreement'),
    tooltip=['language_family:N', 'service:N',
             alt.Tooltip('mean_agreement:Q', format='.2f')],
).properties(width=350, height=420,
    title='Mean cross-variant agreement — language family × LLM service')

alt.Chart(...)

In [13]:
pair_matches = {(a, b): [] for a in VARIANTS for b in VARIANTS if a < b}
for _, row in llm_df.iterrows():
    try:
        vt = ast.literal_eval(str(row['variant_translations']))
    except Exception:
        continue
    for (a, b) in pair_matches:
        ta, tb = vt.get(a), vt.get(b)
        if ta and tb:
            pair_matches[(a, b)].append(
                1 if str(ta).strip().lower() == str(tb).strip().lower() else 0)

pair_rows = []
for (a, b), matches in pair_matches.items():
    if matches:
        sim = round(sum(matches) / len(matches), 3)
        pair_rows += [{'va': a, 'vb': b, 'sim': sim},
                      {'va': b, 'vb': a, 'sim': sim}]
for v in VARIANTS:
    pair_rows.append({'va': v, 'vb': v, 'sim': 1.0})

pair_df = pd.DataFrame(pair_rows)
pair_df['la'] = pair_df['va'].map(VARIANT_LABELS)
pair_df['lb'] = pair_df['vb'].map(VARIANT_LABELS)
label_order = list(VARIANT_LABELS.values())

heat = alt.Chart(pair_df).mark_rect().encode(
    x=alt.X('la:N', sort=label_order, title=None),
    y=alt.Y('lb:N', sort=label_order, title=None),
    color=alt.Color('sim:Q', scale=alt.Scale(scheme='greens', domain=[0, 1]),
                    title='Fraction same'),
    tooltip=['la:N', 'lb:N', alt.Tooltip('sim:Q', format='.2f')],
)
heat_text = heat.mark_text(fontSize=11).encode(
    text=alt.Text('sim:Q', format='.2f'),
    color=alt.condition(alt.datum.sim > 0.6,
                        alt.value('white'), alt.value('black')))
(heat + heat_text).properties(width=300, height=300,
    title='Pairwise prompt-variant similarity (LLM services, normalised match)')

alt.LayerChart(...)

In [14]:
lang_stability = (
    llm_df.groupby(['language_code', 'language_name', 'language_family'])['agreement_rate']
    .mean().reset_index().rename(columns={'agreement_rate': 'mean_llm_agreement'})
    .sort_values('mean_llm_agreement')
)

alt.Chart(lang_stability.head(30)).mark_bar().encode(
    x=alt.X('mean_llm_agreement:Q', scale=alt.Scale(domain=[0, 1]),
            title='Mean LLM agreement rate'),
    y=alt.Y('language_name:N', sort='-x', title=None),
    color=alt.Color('language_family:N', title='Family'),
    tooltip=['language_code:N', 'language_name:N', 'language_family:N',
             alt.Tooltip('mean_llm_agreement:Q', format='.2f')],
).properties(width=400, height=500,
    title='30 least stable languages (mean LLM cross-variant agreement)')

alt.Chart(...)

In [15]:
wiki_df = (detail_df[detail_df['service'] == 'Wikipedia']
           [['language_code', 'best_candidate']].copy())
wiki_df['has_wikipedia'] = (wiki_df['best_candidate'].notna() &
                            (wiki_df['best_candidate'].astype(str).str.strip() != 'nan'))

lang_wiki = lang_stability.merge(wiki_df[['language_code', 'has_wikipedia']],
                                 on='language_code', how='left')
lang_wiki['has_wikipedia']   = lang_wiki['has_wikipedia'].fillna(False)
lang_wiki['wikipedia_label'] = lang_wiki['has_wikipedia'].map({
    True:  'Wikipedia translation exists',
    False: 'No Wikipedia translation',
})
print(lang_wiki.groupby('wikipedia_label')['mean_llm_agreement']
      .agg(['mean', 'median', 'count']).round(3))

base = alt.Chart(lang_wiki)
colours = alt.Scale(
    domain=['Wikipedia translation exists', 'No Wikipedia translation'],
    range=['#2ca02c', '#d62728'],
)
scatter = base.mark_circle(opacity=0.6, size=60).encode(
    x=alt.X('mean_llm_agreement:Q', scale=alt.Scale(domain=[0, 1]),
            title='Mean LLM cross-variant agreement'),
    color=alt.Color('wikipedia_label:N', scale=colours, title=None),
    tooltip=['language_name:N', 'language_family:N',
             alt.Tooltip('mean_llm_agreement:Q', format='.2f'),
             'wikipedia_label:N'],
).properties(width=420, height=220)
strip = base.mark_tick(thickness=2, bandSize=12).encode(
    x=alt.X('mean_llm_agreement:Q', scale=alt.Scale(domain=[0, 1])),
    y=alt.Y('wikipedia_label:N', title=None),
    color=alt.Color('wikipedia_label:N', scale=colours, legend=None),
).properties(width=420, height=80,
             title='LLM stability vs Wikipedia coverage')
alt.vconcat(scatter, strip).resolve_scale(color='shared')

                               mean  median  count
wikipedia_label                                   
No Wikipedia translation      0.430   0.396    840
Wikipedia translation exists  0.728   0.724     40


alt.VConcatChart(...)

## §4 — Across Prompts × Across Services
_Ultimate consensus_

The strongest signal in the pipeline: a translation that appears in multiple (service, variant) combinations simultaneously. A term produced by Claude-minimal *and* Gemini-expert_persona *and* OpenAI-native_rationale is far more credible than one that only appears in one cell of the grid.

**Consensus count** = the number of distinct (service, variant) cells that agree on the single most common translation for a language (out of a maximum of 16: 4 services × 4 variants). **Consensus rate** = consensus count ÷ total cells with data.

In [16]:
consensus_rows = []
for lang_code, grp in scored_df.groupby('language_code'):
    cells = []
    for _, row in grp.iterrows():
        variant = row['prompt_variant']
        for svc, col in LLM_TRANS_COLS.items():
            val = row.get(col)
            if pd.notna(val) and str(val).strip() not in ('', 'nan'):
                cells.append({'service': svc, 'variant': variant,
                              'term': str(val).strip()})
    if not cells:
        continue

    total_cells = len(cells)
    term_counts  = Counter(c['term'] for c in cells)
    top_term, top_count = term_counts.most_common(1)[0]
    agreeing = [c for c in cells if c['term'] == top_term]

    consensus_rows.append({
        'language_code':     lang_code,
        'language_name':     grp['language_name'].iloc[0],
        'language_family':   get_language_family(lang_code),
        'top_term':          top_term,
        'consensus_count':   top_count,
        'total_cells':       total_cells,
        'consensus_rate':    round(top_count / total_cells, 3),
        'n_services_agree':  len(set(c['service'] for c in agreeing)),
        'n_variants_agree':  len(set(c['variant'] for c in agreeing)),
        'n_unique_terms':    len(term_counts),
    })

consensus_df = pd.DataFrame(consensus_rows)

print(f'Languages with LLM data: {len(consensus_df)}')
print(f'Full consensus (rate = 1.0, all cells agree): '
      f'{(consensus_df["consensus_rate"] == 1.0).sum()}')
print(f'Strong consensus (rate ≥ 0.75): '
      f'{(consensus_df["consensus_rate"] >= 0.75).sum()}')
print(f'Weak consensus (rate < 0.5):  '
      f'{(consensus_df["consensus_rate"] < 0.5).sum()}')
print()
print('Service × variant coverage per language (total_cells distribution):')
print(consensus_df['total_cells'].describe().round(1))

Languages with LLM data: 880
Full consensus (rate = 1.0, all cells agree): 1
Strong consensus (rate ≥ 0.75): 9
Weak consensus (rate < 0.5):  789

Service × variant coverage per language (total_cells distribution):
count    880.0
mean      31.0
std        1.3
min       25.0
25%       30.0
50%       32.0
75%       32.0
max       32.0
Name: total_cells, dtype: float64


In [17]:
# ── Distribution of consensus rate ──────────────────────────────────────────
hist = alt.Chart(consensus_df).mark_bar().encode(
    x=alt.X('consensus_rate:Q', bin=alt.Bin(extent=[0, 1], step=0.0625),
            title='Consensus rate (fraction of cells agreeing on top term)'),
    y=alt.Y('count():Q', title='Languages'),
    color=alt.condition(
        alt.datum.consensus_rate >= 0.75,
        alt.value('#2ca02c'), alt.value('#d62728')),
    tooltip=[alt.Tooltip('consensus_rate:Q',
                         bin=alt.Bin(extent=[0,1], step=0.0625)),
             'count():Q'],
).properties(width=420, height=240,
    title='Cross-service × cross-prompt consensus rate distribution')

# ── Breakdown: how many services and variants participate in consensus ────────
svc_bar = alt.Chart(consensus_df).mark_bar().encode(
    x=alt.X('n_services_agree:O', title='Services agreeing on top term'),
    y=alt.Y('count():Q', title='Languages'),
    color=alt.Color('n_services_agree:O',
                    scale=alt.Scale(scheme='blues'), legend=None),
    tooltip=['n_services_agree:O', 'count():Q'],
).properties(width=200, height=200, title='Services in consensus')

var_bar = alt.Chart(consensus_df).mark_bar().encode(
    x=alt.X('n_variants_agree:O', title='Variants agreeing on top term'),
    y=alt.Y('count():Q', title='Languages'),
    color=alt.Color('n_variants_agree:O',
                    scale=alt.Scale(scheme='purples'), legend=None),
    tooltip=['n_variants_agree:O', 'count():Q'],
).properties(width=200, height=200, title='Variants in consensus')

# ── Family breakdown — which families have strong consensus ──────────────────
fam_cons = (
    consensus_df.groupby('language_family')['consensus_rate']
    .mean().reset_index()
    .sort_values('consensus_rate', ascending=False)
)
fam_bar = alt.Chart(fam_cons).mark_bar().encode(
    y=alt.Y('language_family:N',
            sort=alt.EncodingSortField('consensus_rate', order='descending'),
            title=None),
    x=alt.X('consensus_rate:Q', scale=alt.Scale(domain=[0, 1]),
            title='Mean consensus rate'),
    color=alt.condition(
        alt.datum.consensus_rate >= 0.5,
        alt.value('#2ca02c'), alt.value('#d62728')),
    tooltip=['language_family:N',
             alt.Tooltip('consensus_rate:Q', format='.2f')],
).properties(width=300, height=320, title='Mean consensus rate by language family')

hist & (svc_bar | var_bar | fam_bar)

alt.VConcatChart(...)

## §5 — Source Term Pass-Through Analysis

How often does a service simply return "Digital Humanities" (the English source term) rather than providing a translation?

Two complementary metrics:

- **Exact echo rate**: translation string equals the source term exactly (case-insensitive). This matches what the automated `has_source_term` flag already captures.
- **Partial overlap rate**: translation contains "digital" or "humanities" as a substring (case-insensitive) but is *not* an exact echo — this catches cases where the model borrows the English words into an otherwise non-English phrase.

For languages where a service returns the source term exactly, we also inspect whether the accompanying rationale explains *why* (acknowledging untranslatability) or gives a generic rationale — a signal of whether the model is reasoning about the pass-through or just defaulting to it.

In [18]:
from scripts.utils import load_manual_exclusions
from scripts.exploration.explore_confidence_within_variant import load_variant_df

_excl_eval_dir = os.path.join(DATA_DIR, 'translated_terms', TERM_SLUG, 'evaluation')
analysis_langs, search_terms, corrections = load_manual_exclusions(_excl_eval_dir)

SRC_TERM  = TERM   # "Digital Humanities"
SRC_LOWER = SRC_TERM.lower()
SRC_WORDS = {"digital", "humanities"}

# Derive rationale column names from term column names
# e.g. 'claude_translated_term' → 'claude_translation_rationale'
LLM_RAT_COLS = {
    svc: col.replace('_translated_term', '_translation_rationale')
    for svc, col in LLM_TRANS_COLS.items()
}

passthrough_rows = []

for variant in VARIANTS:
    vdf = load_variant_df(DATA_DIR, TERM_SLUG, variant)
    if vdf is None or vdf.empty:
        continue
    vdf = vdf[~vdf['language_code'].isin(analysis_langs)].copy()

    for svc, term_col in LLM_TRANS_COLS.items():
        rat_col = LLM_RAT_COLS.get(svc)
        if term_col not in vdf.columns:
            continue
        for _, row in vdf.iterrows():
            raw = str(row.get(term_col, '') or '').strip()
            if not raw or raw.lower() in ('nan', 'none', ''):
                continue
            exact   = raw.lower() == SRC_LOWER
            words   = set(raw.lower().split())
            partial = (not exact) and bool(words & SRC_WORDS)
            rat_val = str(row.get(rat_col, '') or '').strip() if rat_col and rat_col in vdf.columns else ''
            passthrough_rows.append({
                'language_code':   row['language_code'],
                'language_name':   row.get('language_name', row['language_code']),
                'language_family': row.get('language_family', get_language_family(row['language_code'])),
                'service': svc,
                'variant': variant,
                'translation': raw,
                'exact_echo':     exact,
                'partial_overlap': partial,
                'rationale':      rat_val,
            })

pt_df = pd.DataFrame(passthrough_rows)

total_cells = len(pt_df)
n_exact     = pt_df['exact_echo'].sum()
n_partial   = pt_df['partial_overlap'].sum()

print(f"Total (service × variant × language) cells analysed: {total_cells:,}")
print(f"  Exact echo    (= source term):              {n_exact:,}  ({n_exact/total_cells:.1%})")
print(f"  Partial overlap (DH words, not exact):      {n_partial:,}  ({n_partial/total_cells:.1%})")
print()

rate_df = (
    pt_df.groupby(['service', 'variant'])
    .agg(total=('exact_echo', 'count'),
         exact=('exact_echo', 'sum'),
         partial=('partial_overlap', 'sum'))
    .reset_index()
)
rate_df['exact_rate']   = rate_df['exact']   / rate_df['total']
rate_df['partial_rate'] = rate_df['partial'] / rate_df['total']

print("Exact echo rate by service × variant:")
pivot = rate_df.pivot(index='service', columns='variant', values='exact_rate').round(3)
print(pivot.reindex(columns=VARIANTS).to_string())

Total (service × variant × language) cells analysed: 24,794
  Exact echo    (= source term):              525  (2.1%)
  Partial overlap (DH words, not exact):      2,383  (9.6%)

Exact echo rate by service × variant:
variant   minimal  expert_persona  native_rationale  judge
service                                                   
Claude      0.094           0.015             0.021  0.035
DeepSeek    0.019           0.013             0.014  0.011
Gemini      0.081           0.007             0.014  0.009
Gemma       0.001           0.001             0.003  0.003
Llama       0.007           0.010             0.003  0.000
Mistral     0.005           0.001             0.016  0.028
OpenAI      0.201           0.032             0.027  0.005
Qwen        0.005           0.001             0.008  0.000


In [19]:
# Heatmap: exact echo rate per service × variant
_chart_df = rate_df.copy()
_chart_df['exact_pct'] = (_chart_df['exact_rate'] * 100).round(1)
_chart_df['partial_pct'] = (_chart_df['partial_rate'] * 100).round(1)
_svc_order = sorted(_chart_df['service'].unique())
_var_order  = VARIANTS

heat = alt.Chart(_chart_df).mark_rect().encode(
    x=alt.X('variant:N', sort=_var_order, title='Prompt variant'),
    y=alt.Y('service:N', sort=_svc_order, title=None),
    color=alt.Color('exact_pct:Q',
                    scale=alt.Scale(scheme='orangered', domain=[0, 50]),
                    title='Exact echo %'),
    tooltip=['service:N', 'variant:N',
             alt.Tooltip('exact_pct:Q', title='Exact echo %'),
             alt.Tooltip('partial_pct:Q', title='Partial overlap %'),
             alt.Tooltip('total:Q', title='Cells')],
).properties(width=360, height=240,
    title='Source-term exact echo rate (%) per service × variant')

# Partial overlap heatmap
heat_partial = alt.Chart(_chart_df).mark_rect().encode(
    x=alt.X('variant:N', sort=_var_order, title='Prompt variant'),
    y=alt.Y('service:N', sort=_svc_order, title=None),
    color=alt.Color('partial_pct:Q',
                    scale=alt.Scale(scheme='blues', domain=[0, 20]),
                    title='Partial overlap %'),
    tooltip=['service:N', 'variant:N',
             alt.Tooltip('partial_pct:Q', title='Partial overlap %')],
).properties(width=360, height=240,
    title='Source-term partial overlap rate (%) per service × variant')

(heat | heat_partial).resolve_scale(color='independent')

alt.HConcatChart(...)

In [20]:
# ── Rationale quality for exact-echo cells ───────────────────────────────────
# Does the model explain *why* it's keeping the English term, or give a generic response?
# Simple proxy: rationale mentions "no equivalent", "untranslatable", "no direct",
# "same term", "widely used", "internationally recognised", or language name.

AWARE_PATTERNS = [
    r'no (direct |equivalent |established )?translat',
    r'no (single |established )?equivalent',
    r'untranslat',
    r'same term',
    r'widely used',
    r'international',
    r'adopted.*english',
    r'borrow',
    r'retain',
]
import re as _re
_aware_re = _re.compile('|'.join(AWARE_PATTERNS), _re.IGNORECASE)

echo_df = pt_df[pt_df['exact_echo']].copy()
echo_df['rationale_aware'] = echo_df['rationale'].apply(
    lambda r: bool(_aware_re.search(r)) if r else False
)
echo_df['has_rationale'] = echo_df['rationale'].apply(lambda r: bool(str(r).strip()))

n_echo   = len(echo_df)
n_aware  = echo_df['rationale_aware'].sum()
n_no_rat = (~echo_df['has_rationale']).sum()

print(f"Exact-echo cells: {n_echo}")
print(f"  Rationale explicitly justifies pass-through : {n_aware}  ({n_aware/n_echo:.0%})")
print(f"  No rationale at all                         : {n_no_rat}  ({n_no_rat/n_echo:.0%})")
print(f"  Rationale present but generic               : {n_echo-n_aware-n_no_rat}  ({(n_echo-n_aware-n_no_rat)/n_echo:.0%})")
print()

# By service
aware_by_svc = (
    echo_df.groupby('service')
    .agg(n_echo=('exact_echo','sum'), n_aware=('rationale_aware','sum'))
    .reset_index()
)
aware_by_svc['aware_rate'] = aware_by_svc['n_aware'] / aware_by_svc['n_echo']
print("Aware-rationale rate per service (for exact-echo cells):")
print(aware_by_svc.sort_values('aware_rate', ascending=False)[['service','n_echo','n_aware','aware_rate']].to_string(index=False))

Exact-echo cells: 525
  Rationale explicitly justifies pass-through : 324  (62%)
  No rationale at all                         : 0  (0%)
  Rationale present but generic               : 201  (38%)

Aware-rationale rate per service (for exact-echo cells):
 service  n_echo  n_aware  aware_rate
  Claude     132      110    0.833333
DeepSeek      45       32    0.711111
  OpenAI     191      122    0.638743
  Gemini      85       46    0.541176
   Gemma       6        3    0.500000
   Llama      15        5    0.333333
    Qwen      11        2    0.181818
 Mistral      40        4    0.100000


## §6 — Judge Variant Convergence

The **judge** prompt gives each LLM all other services' minimal-variant translations as context before asking for its own answer. Does this inter-service consultation collapse disagreement — and if so, does it pull everyone toward a specific service's answer?

Two analyses:

**(a) Service × service agreement matrix for judge** — how often do pairs of services produce the same translation in the judge variant? This reveals which services are most "influential" (others agree with them most) and which services cluster with each other.

**(b) Delta from minimal** — for each service pair, how much does judge *increase* agreement compared with minimal? A positive delta means the judge prompt brought two services closer; a negative delta (rare) means it introduced new divergence.

In [21]:
if 'analysis_langs' not in dir():
    from scripts.utils import load_manual_exclusions
    analysis_langs, _, _ = load_manual_exclusions(os.path.join(DATA_DIR, 'translated_terms', TERM_SLUG, 'evaluation'))

from scripts.exploration.explore_confidence_within_variant import load_variant_df

# Load judge and minimal variant DataFrames (analysis-excluded languages removed)
_judge_vdf   = load_variant_df(DATA_DIR, TERM_SLUG, 'judge')
_minimal_vdf = load_variant_df(DATA_DIR, TERM_SLUG, 'minimal')

_svcs = list(LLM_TRANS_COLS.keys())  # title-cased: Claude, OpenAI, …

def _build_svc_term_map(vdf):
    # Returns {language_code: {service: translation_string}}
    vdf = vdf[~vdf['language_code'].isin(analysis_langs)].copy()
    vdf = vdf.drop_duplicates('language_code')
    result = {}
    for _, row in vdf.iterrows():
        lc = row['language_code']
        result[lc] = {}
        for svc, col in LLM_TRANS_COLS.items():
            val = str(row.get(col, '') or '').strip()
            if val and val.lower() not in ('nan', 'none'):
                result[lc][svc] = val.lower()
    return result

judge_map   = _build_svc_term_map(_judge_vdf)   if _judge_vdf   is not None else {}
minimal_map = _build_svc_term_map(_minimal_vdf) if _minimal_vdf is not None else {}

def _pairwise_agreement(term_map):
    # Returns agreement_matrix[svc_a][svc_b] = fraction of languages where both agree
    counts  = {(a, b): 0 for a in _svcs for b in _svcs}
    totals  = {(a, b): 0 for a in _svcs for b in _svcs}
    for lc, svc_terms in term_map.items():
        for a in _svcs:
            for b in _svcs:
                if a in svc_terms and b in svc_terms:
                    totals[(a, b)] += 1
                    if svc_terms[a] == svc_terms[b]:
                        counts[(a, b)] += 1
    result = {}
    for a in _svcs:
        result[a] = {}
        for b in _svcs:
            t = totals[(a, b)]
            result[a][b] = counts[(a, b)] / t if t > 0 else float('nan')
    return result

judge_agree   = _pairwise_agreement(judge_map)
minimal_agree = _pairwise_agreement(minimal_map)

# Build long-form DataFrames for Altair
agree_rows = []
for a in _svcs:
    for b in _svcs:
        j_val = judge_agree[a].get(b, float('nan'))
        m_val = minimal_agree[a].get(b, float('nan'))
        agree_rows.append({
            'svc_a': a, 'svc_b': b,
            'judge_agree': j_val,
            'minimal_agree': m_val,
            'delta': j_val - m_val if (j_val == j_val and m_val == m_val) else float('nan'),
        })
agree_long = pd.DataFrame(agree_rows)

print("Judge variant — mean off-diagonal agreement:",
      round(agree_long[agree_long['svc_a'] != agree_long['svc_b']]['judge_agree'].mean(), 3))
print("Minimal variant — mean off-diagonal agreement:",
      round(agree_long[agree_long['svc_a'] != agree_long['svc_b']]['minimal_agree'].mean(), 3))
print()
# Which service has the highest mean agreement with all others in judge?
svc_influence = (
    agree_long[agree_long['svc_a'] != agree_long['svc_b']]
    .groupby('svc_b')['judge_agree'].mean()
    .sort_values(ascending=False)
    .rename('mean_agreement_from_others')
)
print("Mean agreement FROM others in judge (higher = more influential):")
print(svc_influence.round(3).to_string())

Judge variant — mean off-diagonal agreement: 0.274
Minimal variant — mean off-diagonal agreement: 0.061

Mean agreement FROM others in judge (higher = more influential):
svc_b
OpenAI      0.335
DeepSeek    0.319
Gemma       0.302
Claude      0.293
Gemini      0.258
Mistral     0.233
Qwen        0.230
Llama       0.221


In [22]:
_svc_order = _svcs  # consistent ordering

def _heatmap(df, val_col, title, scheme, domain):
    return alt.Chart(df).mark_rect().encode(
        x=alt.X('svc_a:N', sort=_svc_order, title='Service'),
        y=alt.Y('svc_b:N', sort=_svc_order, title=None),
        color=alt.Color(f'{val_col}:Q',
                        scale=alt.Scale(scheme=scheme, domain=domain),
                        title=val_col),
        tooltip=['svc_a:N', 'svc_b:N',
                 alt.Tooltip(f'{val_col}:Q', format='.2f')],
    ).properties(width=300, height=280, title=title)

heat_judge   = _heatmap(agree_long, 'judge_agree',   '(a) Judge — agreement matrix',   'redyellowgreen', [0, 1])
heat_minimal = _heatmap(agree_long, 'minimal_agree', '(a) Minimal — agreement matrix', 'redyellowgreen', [0, 1])
heat_delta   = _heatmap(agree_long, 'delta',         '(b) Delta: judge − minimal',      'blueorange',    [-0.3, 0.3])

(heat_minimal | heat_judge | heat_delta).resolve_scale(color='independent')

alt.HConcatChart(...)

In [23]:
if 'analysis_langs' not in dir():
    from scripts.utils import load_manual_exclusions
    analysis_langs, _, _ = load_manual_exclusions(os.path.join(DATA_DIR, 'translated_terms', TERM_SLUG, 'evaluation'))

# ── Service agreement by language family (judge variant) ────────────────────
# For each language family, which service pairs agree most often in judge?
# Collapsed to: mean pairwise agreement per family × off-diagonal pairs.

_judge_vdf2 = load_variant_df(DATA_DIR, TERM_SLUG, 'judge')
if _judge_vdf2 is not None:
    _judge_vdf2 = _judge_vdf2[~_judge_vdf2['language_code'].isin(analysis_langs)].copy()
    _judge_vdf2 = _judge_vdf2.drop_duplicates('language_code')
    _judge_vdf2['language_family'] = _judge_vdf2['language_code'].apply(get_language_family)

    fam_agree_rows = []
    for fam, grp in _judge_vdf2.groupby('language_family'):
        fam_map = {}
        for _, row in grp.iterrows():
            lc = row['language_code']
            fam_map[lc] = {}
            for svc, col in LLM_TRANS_COLS.items():
                val = str(row.get(col, '') or '').strip()
                if val and val.lower() not in ('nan', 'none'):
                    fam_map[lc][svc] = val.lower()
        fam_agree = _pairwise_agreement(fam_map)
        off_diag = [(fam_agree[a][b]) for a in _svcs for b in _svcs
                    if a != b and a in fam_agree and b in fam_agree[a]
                    and fam_agree[a][b] == fam_agree[a][b]]
        if off_diag:
            fam_agree_rows.append({'language_family': fam,
                                   'mean_judge_agreement': sum(off_diag)/len(off_diag),
                                   'n_languages': len(grp)})

    fam_agree_df = pd.DataFrame(fam_agree_rows).sort_values('mean_judge_agreement', ascending=False)
    print("Mean pairwise judge agreement by language family (top 15):")
    print(fam_agree_df.head(15).to_string(index=False))

Mean pairwise judge agreement by language family (top 15):
         language_family  mean_judge_agreement  n_languages
      Armenian languages              1.000000            1
       Japonic languages              0.464286            1
 Indo-European languages              0.403955          229
     Dravidian languages              0.376984            9
        Basque languages              0.357143            1
     Tai-Kadai languages              0.330357            8
       Khoisan languages              0.321429            1
        Altaic languages              0.305804           32
  Afro-Asiatic languages              0.305346           44
  Austronesian languages              0.294539           78
  Sino-Tibetan languages              0.275076           47
        Language isolate              0.267857            8
Austro-Asiatic languages              0.258929           12
    Artificial languages              0.230952           15
     Caucasian languages              0.2